In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
%sql

WITH category_counts_per_listing AS(
    SELECT
        listing_id,
        category,
        COUNT(*) AS count
    FROM airbnb_reviews_gold
    LATERAL VIEW EXPLODE(review_categories) AS category
    GROUP BY listing_id, category
    ORDER BY listing_id, count
),
all_category_counts_and_listings AS(
  SELECT
      h.id AS id,
      FIRST(h.host_name) AS name,
      l.id as listing_id,
      FIRST(l.name) as listing_name,
      ARRAY_AGG(STRUCT(c.category, c.count)) as categories_counts
  FROM airbnb_hosts_silver h
  JOIN airbnb_listings_silver l
  ON h.id = l.host_id
  JOIN category_counts_per_listing c
  ON l.id = c.listing_id
  GROUP BY h.id, l.id
)
SELECT
  id,
  name,
  listing_id,
  listing_name,
  SLICE(TRANSFORM(array_sort(
    categories_counts,
    (x, y) -> CASE WHEN x.count < y.count THEN -1
                   WHEN x.count > y.count THEN 1
                   ELSE 0
              END
  ), x-> x.category), 1, 5) AS top_5_categories
FROM all_category_counts_and_listings
ORDER BY id
  